## Tactical Development


### Dataset :: (fake dataset - databricks market place)

https://docs.databricks.com/aws/en/compute/configure

In [0]:
# %sql
# Sample Script 
# CREATE SCHEMA IF NOT EXISTS medallion_catalog.db_bronze
# MANAGED LOCATION 'abfss://catalog@container.dfs.core.windows.net/folder';

# USE CATALOG medallion_catalog;
# USE SCHEMA db_bronze;

# DROP TABLE IF EXISTS medallion_catalog.db_bronze.companies;

# CREATE TABLE IF NOT EXISTS medallion_catalog.db_bronze.companies 
# AS
# SELECT company_name, founded_date, country
#   FROM read_files('/Volumes/medallion_catalog/db_landing/vol_medallion/top_tech_companies/',
#   format => 'csv',
#   header => true);


## V03
 


##### Databricks built-in spark session 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# 1. Configuration for Cloud/K8s Scalability
# Note: On Databricks, you usually enable 'Autoscaling' in the Cluster UI.
# This script ensures the software knows how to distribute that load.
spark.conf.set("spark.databricks.cloudFiles.optimizePerformance", "true")

# Define the Schema for your fake JSON
json_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("value", DoubleType(), True),
    StructField("metadata", StringType(), True)
])

# 2. Setup the Auto-loader (Source Diversity & Flexibility)
# 'cloudFiles' uses AWS SNS/SQS or Directory Listing to find files
s3_path = "s3://your-bucket-name/incoming_json_data/"
checkpoint_path = "s3://your-bucket-name/checkpoints/streaming_job_01"

streaming_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    # FLEXIBILITY: This controls the 'size' perceived by the cluster.
    # If a file is 1GB and this is 128MB, Spark will split the work across executors.
    .option("maxBytesPerTrigger", "128mb") 
    .schema(json_schema)
    .load(s3_path))

# 3. Transformation Logic
processed_df = streaming_df.withColumn("ingested_at", current_timestamp())

# 4. The Sink (Writing the data)
query = (processed_df.writeStream
    .format("delta")  # Using Delta Lake for performance
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    # TRIGGER: Ensures the cluster checks for files every 1 second
    .trigger(processingTime='1 second') 
    .start("dbfs:/mnt/processed_table"))

# Keep the session alive
# query.awaitTermination()


#### business_daily_events

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("campaign_id", StringType(), True),
    StructField("carrier", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("city", StringType(), True),
    StructField("clicks", LongType(), True),
    StructField("conversions", LongType(), True),
    StructField("destination_region", StringType(), True),
    StructField("event_group", StringType(), True),
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("impressions", LongType(), True),
    StructField("num_packages", LongType(), True),
    StructField("opened_by_employee_id", StringType(), True),
    StructField("region", StringType(), True),
    StructField("spend_usd", DoubleType(), True),
    StructField("store_id", StringType(), True),
    StructField("subsidiary_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("warehouse_id", StringType(), True)
])

In [0]:
customers_df = (
    spark.readStream
    .format("json")
    .schema(schema)
    .load("/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/business_daily_events/")
)

##### New columns:: file_path, ingestion_date

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_transformed_df = (
    customers_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

##### writeStream


In [0]:
streaming_query = (
    customers_transformed_df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/business_daily_events/_checkpoint_stream")
        .trigger(availableNow=True)
        .toTable("end_to_end_retail.db_bronze.business_daily_events")
)

streaming_query.awaitTermination()

In [0]:
streaming_query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_bronze.business_daily_events limit 10

If new file is dropped in the volume while streaming is listening to the volume, table will be updated;

In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v02/business_daily_events/retail_business_events_2025-11-01.json")
display(df.head(5))

#### customer_changes_daily

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

customer_schema = StructType([
    StructField("city", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("source_subsidiary", StringType(), True),
    StructField("timestamp", StringType(), True)
])

In [0]:
customers_df = (
    spark.readStream
    .format("json")
    .schema(customer_schema)
    .load("/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/customer_changes_daily/")
)

In [0]:
customers_transformed_df = (
    customers_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
streaming_query = (
    customers_transformed_df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/customer_changes_daily/_checkpoint_stream")
        .trigger(availableNow=True)
        .toTable("end_to_end_retail.db_bronze.customer_changes_daily")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_bronze.customer_changes_daily limit 5

In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-01.json")
display(df.head(5))

#### subsidiary_daily_orders

##### bright_home_orders

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

subsidiary_orders_schema = StructType([
    StructField("subsidiary_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("order_timestamp", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("region", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("category", StringType(), True),
    StructField("qty", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount_pct", StringType(), True),
    StructField("coupon_code", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("order_date", StringType(), True)
])

In [0]:
bright_home_orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(subsidiary_orders_schema)
    .load("/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/subsidiary_daily_orders/bright_home_orders/")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

bright_home_orders_transformed_df = (
    bright_home_orders_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
streaming_query = (
    bright_home_orders_transformed_df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/subsidiary_daily_orders/bright_home_orders/_checkpoint_stream")
        .trigger(availableNow=True)
        .toTable("end_to_end_retail.db_bronze.bright_home_orders")
)

streaming_query.awaitTermination()

In [0]:
streaming_query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_bronze.bright_home_orders limit 5

In [0]:
# Load CSV file
df = spark.read.csv("/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders/bright_home_orders/bsh_orders_2025-11-01.csv", header=True)
display(df.limit(5))

##### lumina_sports_orders

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

subsidiary_orders_schema = StructType([
    StructField("subsidiary_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("order_timestamp", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("region", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("category", StringType(), True),
    StructField("qty", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount_pct", StringType(), True),
    StructField("coupon_code", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("order_date", StringType(), True)
])

In [0]:
lumina_sports_orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(schema)
    # .load("/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/subsidiary_daily_orders/lumina_sports_orders/")
    .load("/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/subsidiary_daily_orders/lumina_sports_orders/")
  
    )

In [0]:
from pyspark.sql.functions import col, current_timestamp

lumina_sports_orders_transformed_df = (
    lumina_sports_orders_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
streaming_query = (
    lumina_sports_orders_transformed_df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/subsidiary_daily_orders/lumina_sports_orders/_checkpoint_stream")
        .trigger(availableNow=True)
        .toTable("end_to_end_retail.db_bronze.lumina_sports_orders")
)

streaming_query.awaitTermination()

In [0]:
streaming_query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_bronze.lumina_sports_orders limit 5

In [0]:
# Load CSV file
df = spark.read.csv("/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders/lumina_sports_orders/lms_orders_2025-11-01.csv", header=True)
display(df.limit(5))


##### northstar_outfitters_orders

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

subsidiary_orders_schema = StructType([
    StructField("category", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("coupon_code", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("discount_pct", LongType(), True),
    StructField("order_date", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("order_timestamp", StringType(), True),
    StructField("qty", LongType(), True),
    StructField("region", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("subsidiary_id", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("unit_price", DoubleType(), True)
])

In [0]:
northstar_outfitters_orders_df =(
    spark.readStream
    .format("json")
    .schema(subsidiary_orders_schema)
    .load("/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/subsidiary_daily_orders/northstar_outfitters_orders/")
)

In [0]:
northstar_outfitters_orders_transformed_df = (
    northstar_outfitters_orders_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
streaming_query = (
    customers_transformed_df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/end_to_end_retail/db_landing/managed_spark_autoloader/subsidiary_daily_orders/northstar_outfitters_orders/_checkpoint_stream")
        .trigger(availableNow=True)
        .toTable("end_to_end_retail.db_bronze.northstar_outfitters_orders")
)

streaming_query.awaitTermination()

In [0]:
streaming_query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_bronze.northstar_outfitters_orders limit 5

In [0]:
# Load json file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders/northstar_outfitters_orders/nso_orders_2025-11-01.json")
display(df.limit(5))
